In [ ]:
# 将语料库分片
import os
import shutil

file_path = 'data/wiki1m_for_simcse.txt'
output_dir = 'data/wiki1m_for_simcse_splited'
chunks = 10

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.mkdir(output_dir)

with open(file_path, 'r', encoding='utf-8') as file:
    for i, line in enumerate(file):
        output_file_path = os.path.join(output_dir, str(i % chunks) + '.txt')
        with open(output_file_path, 'a', encoding='utf-8') as output_file:
            output_file.write(line)

## spacy记录

不同模型有不同精度

#### [en_core_web_sm](https://spacy.io/models/en#en_core_web_sm)

支持的实体类型

<img src="https://lyu-oss.oss-cn-beijing.aliyuncs.com/img-bed/image-20231227161004504.png" alt="image-20231227161004504" style="zoom:50%;" />

* CARDINAL -- 数字 【'九百多', '8000', '八百'，'1111.01'】
* DATE -- 大粒度时间，时间段 【 '今年', '明天', '今天', '国庆期间', '3天', '10天'， '三年前'】
* **EVENT -- 事件 【'伦敦奥运会', '世界杯','第14届中国国际工业博览会', '深圳市五届人大二次会议'】**
* **FAC -- 小地点 【'轻轨1号线锡北运河站', '万达广场', '乐购超市'，'永盛大酒店', '110岗亭'】**
* **GPE -- 地点 【'美国', '加拿大', '北京', '中国',】**
* **LANGUAGE -- 语言 【'英语', '汉语', '上海话', '中文'】**
* **LAW -- 规章制度 【'青少年犯罪法', '阿鲁巴决议', '反托拉斯法'】**
* **LOC -- 大地点 【'欧洲人', '欧洲', '亚洲', '天山山脉', '巴尔斯卡乌尼河''月球', '火星'】**
* MONEY -- 货币 【'￥8000', '9200', '60元', '3000美金'】部分省略货币单位也能识别
* **NORP -- 人物、地点 【'德国', '中国', '中国人','韩版', '日媒', '日本', '扬州'】**
* ORDINAL -- 顺序 【'首', '第六', '第二', '第一','第几'】
* **ORG -- 组织 【'杭州江干区公安分局'，'LG', '三星', '苹果'，'中国移动', '央行'，'中央社'， '外交部'】**
* PERCENT -- 比率 【'0.2%', '0.2%', '48%'】
* **PERSON -- 人物 【'栾丽娜', '斯蒂芬·弗雷斯', '栗元广', '小雨', '小银狐'】**
* **PRODUCT -- 产品 【'Android', 'iOS'， '金龙鱼', 'UCWeb的浏览器'】**
* QUANTITY -- 量级 【'87,000', '46.522吨', '48248.8千克', '0.23点'，'5.3级', '4200公里' 】
* TIME -- 时间 【'十分钟', '下午', '七点', '今晚'】
* **WORK_OF_ART -- 艺术品 【'刺客信条', '富春山居图', '有一说一'，'探索•发现', '定窑考工记'】**

事件、地点、语言、规章制度、人物、组织、产品、艺术品

In [ ]:
#分词统计
# 统计语料库中各种类型的实体数量

# 事件、地点、语言、规章制度、人物、组织、产品、艺术品
import spacy
from tqdm import tqdm
from collections import Counter

type_list = ['EVENT', 'FAC', 'GPE', 'LANGUAGE', 'LAW', 'LOC', 'NORP', 'ORG', 'PERSON', 'PRODUCT', 'WORK_OF_ART']

nlp = spacy.load("en_core_web_sm")

file_path = 'data/wiki1m_for_simcse.txt'

type_counts = Counter()

with open(file_path, 'r', encoding='utf-8') as file:
    for index, line in tqdm(enumerate(file)):
        words = line.strip().split()
        text = ' '.join(words)
        doc = nlp(text)
        for ent in doc.ents:
            if ent.label_ in type_list:
                type_counts[ent.label_] += 1
    

for type_name, count in type_counts.items():
    print(f"{type_name}: {count}")
# GPE: 373869
# NORP: 152530
# ORG: 532072
# WORK_OF_ART: 80636
# FAC: 37106
# PERSON: 505931
# LOC: 56057
# PRODUCT: 22275
# LAW: 6742
# EVENT: 29668
# LANGUAGE: 8100
# sum: 1,763,006
    
# 事件:P31: Q1656682
# 地点:P31: Q2221906
# 语言:P31: Q34770
# 规章制度:P31: Q22097341
# 人物:P31: Q5
# 组织:P31: Q43229
# 产品:P31: Q2424752
# 艺术品:P31: Q838948

In [ ]:
# 随机选句子
import random
from tqdm import tqdm

# LANG_LIST = ['de', 'en', 'es', 'fr', 'it', 'nl', 'pl', 'pt', 'ru', 'zh']
LANG_LIST = ['zh']
input_file = '../data/wiki_all_sent_{lang}.txt'
output_file_template = '../Dataset-LyuCSE/wiki1m_{lang}.txt'

sample_num = 1_000_000
for lang in tqdm(LANG_LIST):
    sent_list = []
    with open(input_file.format(lang=lang), 'r', encoding='utf-8') as file:
        for line in file:
            sent_list.append(line.strip())
    sample_list = random.sample(sent_list, sample_num)
    output_file = output_file_template.format(lang=lang)
    with open(output_file, 'w', encoding='utf-8') as file:
        for sent in sample_list:
            file.write(sent + '\n')


In [ ]:
# 生成训练数据
"""策略：
同一个page的作为一个batch，其中一定比例的句子会被随机替换成其他page的句子
"""
import random
from tqdm import tqdm
from backend import MySQLClient
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt')

all_sent_file = '../data/wiki_all_sent_NEWS.txt'
MySQL = MySQLClient()
batch_size = 64
sent_num = 1_000_000
batch_num = sent_num // batch_size

batch_sent_list = []
offset = 0
domain = 'NEWS'
limit = 1000

page_dropout_rate = 0.2 # 可以控制page的多样性
sent_dropout_rate = 0.9 # 可以控制一篇文章内句子的多样性

output_file = f'../Dataset-LyuCSE/t_wiki1m_page_with_batch_dropout{sent_dropout_rate}.txt'

cnt = 0
pbar = tqdm(total=batch_num)
while cnt < batch_num:
    page_content_list = MySQL.get_wiki_page_content(domain, offset)
    if not page_content_list:
        raise Exception("no more page content")
    pbar.set_postfix_str(f'offset:{offset}')
    offset += limit
    page_content_list = [content for id, content in page_content_list.items()]  # dict to list
    # 丢弃部分page
    page_content_list = random.sample(page_content_list, int(len(page_content_list) * (1 - page_dropout_rate)))

    sent_list = []  # bs大小
    for page_content in page_content_list:
        all_sent = sent_tokenize(page_content)
        # 如果句子数量不足，跳过这个page
        if len(all_sent) < batch_size:
            continue
        # 采样bs个
        sent_list = random.sample(all_sent, batch_size)
        assert len(sent_list) == batch_size
        batch_sent_list.append(sent_list)
        cnt += 1
        pbar.update(1)
        if cnt >= batch_num:
            break
pbar.close()

assert len(batch_sent_list) == batch_num

if sent_dropout_rate > 0:
    # 加载所有句子库
    print(f"loading all sent from {all_sent_file}")
    with open(all_sent_file, 'r', encoding='utf-8') as f:
        all_sent_list = f.read().splitlines()

    new_batch_sent_list = []
    for sent_list in batch_sent_list:
        dropout_num = int(sent_dropout_rate * batch_size)   # 随机丢弃的句子数量
        # 原batch 采样出bs-dropout_num个句子
        new_sent_list = random.sample(sent_list, batch_size - dropout_num)
        # 从所有句子库中采样出dropout_num个句子
        new_sent_list += random.sample(all_sent_list, dropout_num)
        assert len(new_sent_list) == batch_size
        new_batch_sent_list.append(new_sent_list)
    batch_sent_list = new_batch_sent_list

print(f"start writing to {output_file}")
with open(output_file, 'w', encoding='utf-8') as f:
    for sent_list in batch_sent_list:
        for sent in sent_list:
            f.write(sent + '\n')


In [3]:
# 采样
import random
from tqdm import tqdm

source_file = '../data/wiki_all_sent_chemistry.txt'
output_file = '../Dataset-LyuCSE/wiki1m_chemistry.txt'

sample_num = 1_000_000
seed = 42

sent_list = []
with open(source_file, 'r', encoding='utf-8') as file:
    sent_list = file.read().splitlines()

random.seed(seed)
sample_list = random.sample(sent_list, sample_num)

with open(output_file, 'w', encoding='utf-8') as file:
    for sent in sample_list:
        file.write(sent + '\n')